In [1]:
from pathlib import Path 

import numpy as np
import pandas as pd
import sqlalchemy as db
from tqdm.auto import tqdm
from qdrant_client import QdrantClient 
from qdrant_client.models import Distance, VectorParams, PointStruct
from qdrant_client.http import models

/home/amos/anaconda3/envs/face/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CLIENT = QdrantClient(host='192.168.0.131', port=6333)  

In [3]:
collections = [x.name for x in CLIENT.get_collections().collections]
if 'FacialEmbeddings' not in collections:
        CLIENT.recreate_collection(collection_name='FacialEmbeddings',
                                   vectors_config=VectorParams(size=128, distance=Distance.COSINE))

In [4]:
username = 'amos'
password = 'M0$hicat'
host = '192.168.0.131'
port = '3306'
database = 'CineFace'

In [5]:
connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
engine = db.create_engine(connection_string)
conn = engine.connect()

In [11]:
df = pd.read_csv("/home/amos/programs/CineFace/data/faces/king-of-the-hill_1997_118375/King.of.the.Hill.S09E15.It.Ain't.Over.'til.the.Fat.Neighbor.Sings.480p.DVDRip.10bit.x265.HEVC.DD2.0-PHOCiS.csv", index_col=0)
df.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,img_height,filepath,encoding,series_id,episode_id,filename,distance_from_center,pct_of_frame,season,episode
0,0.568,0.271,0.617,0.365,0.575,0.311,0.595,0.310,0.581,0.330,...,480,/home/amos/media/tv/King.of.the.Hill.1997.S01-...,-0.109785\n0.061774\n0.0964519\n0.0913509\n-0....,118375,NaN,King.of.the.Hill.S09E15.It.Ain't.Over.'til.the...,110.00,0.46,9,15
1,0.570,0.273,0.618,0.367,0.577,0.309,0.597,0.310,0.582,0.330,...,480,/home/amos/media/tv/King.of.the.Hill.1997.S01-...,-0.069378\n0.0285719\n0.0634699\n0.097021\n-0....,118375,NaN,King.of.the.Hill.S09E15.It.Ain't.Over.'til.the...,109.81,0.44,9,15
2,0.567,0.271,0.616,0.365,0.573,0.310,0.594,0.310,0.579,0.330,...,480,/home/amos/media/tv/King.of.the.Hill.1997.S01-...,-0.0969313\n0.0352653\n0.0741248\n0.084752\n-0...,118375,NaN,King.of.the.Hill.S09E15.It.Ain't.Over.'til.the...,109.40,0.46,9,15
3,0.208,0.283,0.249,0.375,0.227,0.319,0.245,0.318,0.242,0.333,...,480,/home/amos/media/tv/King.of.the.Hill.1997.S01-...,-0.0805755\n0.0647411\n0.0514798\n-0.0313128\n...,118375,NaN,King.of.the.Hill.S09E15.It.Ain't.Over.'til.the...,211.93,0.37,9,15
4,0.440,0.202,0.570,0.475,0.485,0.302,0.546,0.307,0.522,0.348,...,480,/home/amos/media/tv/King.of.the.Hill.1997.S01-...,-0.0742334\n-0.00739619\n0.00313161\n-0.014724...,118375,NaN,King.of.the.Hill.S09E15.It.Ain't.Over.'til.the...,78.06,3.53,9,15


In [8]:
def parse_vector(vector):
    return np.array([float(x) for x in vector.split('\n')])

In [15]:
row = df.iloc[0]
encoding = parse_vector(row['encoding'])
encoding

array([-0.109785  ,  0.061774  ,  0.0964519 ,  0.0913509 , -0.0943675 ,
       -0.0933151 , -0.0348004 , -0.145592  ,  0.0399742 , -0.100771  ,
        0.159844  , -0.00107771, -0.186074  , -0.0265654 , -0.027095  ,
        0.108819  , -0.0886119 , -0.0501873 , -0.178698  , -0.0965923 ,
        0.0570179 ,  0.0665161 ,  0.0643479 , -0.0375572 , -0.132478  ,
       -0.244406  , -0.0849263 , -0.0292872 ,  0.0537057 , -0.098721  ,
        0.0842916 ,  0.0644786 , -0.166744  ,  0.0276757 , -0.00952773,
        0.10546   , -0.093795  , -0.0686596 ,  0.264365  , -0.0639599 ,
       -0.20522   ,  0.0689494 ,  0.0757074 ,  0.190823  ,  0.203695  ,
       -0.0734585 ,  0.0202858 , -0.0483937 ,  0.0544617 , -0.237625  ,
        0.00543259,  0.167105  ,  0.0608184 ,  0.0806621 ,  0.106518  ,
       -0.125076  ,  0.01118   ,  0.140999  , -0.120231  ,  0.0713697 ,
        0.0882722 , -0.112517  , -0.048873  ,  0.0110681 ,  0.0708649 ,
        0.0638212 , -0.0821231 , -0.150428  ,  0.160165  , -0.20

In [20]:
# df['encoding'] = df['encoding'].map(parse_vector)

CLIENT.upload_points(collection_name='FacialEmbeddings',
              points=[
              PointStruct(
                id=0,
                vector=encoding,
                payload={'series_id': str(row['series_id']),
                         'episode_id': str(row['episode_id']),
                         'frame_num': str(row['frame_num']),
                         'face_num': str(row['face_num'])}
            )
            ]
)

In [21]:
cnt = CLIENT.count(collection_name='FacialEmbeddings').count
vectors = CLIENT.retrieve('FacialEmbeddings', [x for x in range(cnt)], with_vectors=True, with_payload=True)
vectors[0]

Record(id=0, payload={'episode_id': 'nan', 'face_num': '0', 'frame_num': '48', 'series_id': '118375'}, vector=[-0.08350843, 0.04698866, 0.07336655, 0.069486454, -0.07178104, -0.070980534, -0.026471073, -0.110745184, 0.030406548, -0.07665189, 0.12158602, -0.0008197647, -0.141538, -0.020207087, -0.020609926, 0.08277364, -0.06740302, -0.038175184, -0.1359274, -0.07347335, 0.04337091, 0.050595757, 0.048946507, -0.028568044, -0.10076996, -0.18590847, -0.06459955, -0.022277435, 0.04085147, -0.07509255, 0.06411677, 0.04904592, -0.12683453, 0.02105164, -0.0072473083, 0.080218606, -0.07134557, -0.052226223, 0.20109037, -0.048651367, -0.15610147, 0.052446656, 0.057587158, 0.14515033, 0.15494147, -0.055876523, 0.015430481, -0.036810875, 0.041426525, -0.18075047, 0.004132323, 0.12710914, 0.046261776, 0.061355975, 0.08102337, -0.09513959, 0.008504115, 0.107251495, -0.09145423, 0.05428767, 0.067144625, -0.08558654, -0.037175458, 0.008418998, 0.05390369, 0.048545867, -0.06246729, -0.1144237, 0.121830

In [26]:
dict(vectors[0].payload)

{}

In [129]:
(10000000 * 1.5 * 128 * 4)/1024/1024/1024

7.152557373046875